In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
from PIL import Image
from pathlib import Path
import os


In [ ]:
# papermill will inject `input_image`. Provide a default for local debugging.
try:
    input_image
except NameError:
    input_image = "test.png"  # replace with a local test image when debugging
print("Input image:", input_image)


In [ ]:
# Model path (ensure models/best_model.pth exists in your repo)
model_path = "models/best_model.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)
# If your saved model is a state_dict, change loading accordingly.
model = torch.load(model_path, map_location=device)
model.to(device)
model.eval()
print("Loaded model:", model_path)


In [ ]:
# Read image and preprocess (modify Resize if your model uses other size)
img = Image.open(input_image).convert("RGB")

transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor()
])
x = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)

# Convert model output to a mask array. Adjust logic if your model outputs differ.
arr = pred.squeeze().cpu().numpy()
if arr.ndim == 4:
    # e.g., [B, C, H, W]
    arr = arr.argmax(axis=1).squeeze()
elif arr.ndim == 3 and arr.shape[0] > 1:
    # [C, H, W] -> per-pixel class id
    arr = arr.argmax(axis=0)

# For binary logits/probabilities -> threshold
if arr.max() <= 1:
    mask = (arr > 0.5).astype('uint8') * 255
else:
    mask = arr.astype('uint8')

print('Mask shape:', mask.shape, 'unique labels:', np.unique(mask))


In [ ]:
out_dir = Path("notebooks/output_oct")
out_dir.mkdir(parents=True, exist_ok=True)
mask_name = f"mask_{Path(input_image).stem}.png"
mask_path = out_dir / mask_name

from PIL import Image as PILImage
PILImage.fromarray(mask).save(mask_path)
print('Saved mask to', mask_path)
mask_path
